# Creating Size-Dependent States and Operators

The [State](../apidoc/_autosummary/pulser.backend.State.rst) and [Operator](../apidoc/_autosummary/pulser.backend.Operator.rst) classes let you build arbitrary state and operator objects from scratch using the `from_state_amplitudes()` and `from_operator_repr()` methods (as shown in [Execution on an Emulator](backends.nblink) and [Results and Observables](../results.ipynb)). These objects are used to create initial states, calculate observables such as `Fidelity`, and find expectation values of custom operators at the end of your Pulser simulation.

A common use case for these is many-body physics simulations, where you often want to:

- prepare reference initial states that are non-trivial,
- scale the number of qubits `n_qubits` to look for trends that persist (or not).

This page presents some simple recipes for defining commonly used states and operators with a variable system size.

## Size-dependent helper function for Pauli operators

For a two-level system with `eigenstates = ("r", "g")` (corresponding to the [ground-rydberg basis](../conventions.md#pauli-matrix-form)), the Pauli matrices are written as:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pulser.backend import State, Operator, StateRepr, OperatorRepr
from pulser_simulation import QutipConfig

eigenstates = ("r", "g")

sigma_repr = {
    "x": {"gr": 1.0, "rg": 1.0},
    "y": {"gr": 1.0j, "rg": -1.0j},
    "z": {"rr": 1.0, "gg": -1.0},
}

Instead of repeating `eigenstates` and `n_qudits` at every call, it is convenient to write small helper functions parametrized by the number of qubits `n_qubits`. The `multi_pauli_op` function below builds the operator for a product of identical Pauli matrices acting on an arbitrary subset of sites (e.g. `multi_pauli_op(n_qubits=4, operator_class=OperatorRepr, pauli="x", idxs=[0, 2])` builds $X_0 X_2$), and `get_state` builds a few reference states of size `n_qubits`. `QutipConfig` has been used below as an example for the backend config class.

In [ ]:
def multi_pauli_op(
    n_qubits, operator_class=OperatorRepr, pauli="x", idxs=0
) -> Operator:
    """Builds the operator for a Pauli string, e.g.
    multi_pauli_op(n_qubits=4, operator_class=OperatorRepr, pauli="x", idxs=[0, 2]) builds X_0 X_2.
    Does not support mixed Pauli operators, such as X_0 Z_1.
    """
    # a single int and lists/tuples/sets of ints are both supported as idxs
    # sets are used to remove duplicates
    if isinstance(idxs, int):
        idx = {idxs}
    else:
        idx = set(idxs)

    return operator_class.from_operator_repr(
        eigenstates=eigenstates,
        n_qudits=n_qubits,
        operations=[(1.0, [(sigma_repr[pauli], idx)])],
    )

In [ ]:
def get_state(n_qubits, state_class=StateRepr, state_name="plus") -> State:
    """Builds few reference states of size n_qubits.
    """
    if state_name == "plus":
        # |+>^n_qubits : uniform superposition of all basis states
        dim = 2**n_qubits
        amplitudes = {
            "".join(
                eigenstates[int(b)]
                for b in format(i, f"0{n_qubits}b")
                # this converts the integer i into a binary string of a fixed length (n_qubits)
            ): 1.0
            / np.sqrt(dim)
            for i in range(dim)
        }
        return state_class.from_state_amplitudes(
            eigenstates=eigenstates, amplitudes=amplitudes
        )

    elif state_name == "ghz":
        # (|rr...r> + |gg...g>) / sqrt(2)
        norm = 1.0 / np.sqrt(2)
        return state_class.from_state_amplitudes(
            eigenstates=eigenstates,
            amplitudes={"r" * n_qubits: norm, "g" * n_qubits: norm},
        )

    else:
        raise NotImplementedError(f"State {state_name} not implemented.")

`multi_pauli_op` and `get_state` both only depend on the `n_qubits` argument, so the exact same
code can be reused to build operators and states for any system size, and
they are built directly with `from_operator_repr()` /
`from_state_amplitudes()`, so they remain compatible with remote backends.

In [ ]:
operator_class = QutipConfig.operator_type

x0 = multi_pauli_op(n_qubits=4, operator_class=operator_class, pauli="x", idxs=0)  # X_0
z1z3 = multi_pauli_op(
    n_qubits=4, operator_class=operator_class, pauli="z", idxs=[1, 3]
)  # Z_1 Z_3, indices as a list also works

z1z3.to_qobj()

## Building reference states

`get_state()` builds reference states via `from_state_amplitudes()`. Below, the `"plus"` and `"ghz"` states are built for the same `n_qubits` and compared by their bitstring probabilities, obtained with `bitstring_probabilities()`.

In [ ]:
state_class = QutipConfig.state_type

plus_state = get_state(n_qubits=4, state_class=state_class, state_name="plus")
ghz_state = get_state(n_qubits=4, state_class=state_class, state_name="ghz")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, state, title in zip(axes, (plus_state, ghz_state), ("|+>^n_qubits", "GHZ")):
    probs = state.bitstring_probabilities()
    ax.bar(probs.keys(), probs.values())
    ax.set_title(title)
    ax.set_xlabel("bitstring")
    ax.tick_params(axis="x", rotation=90)

axes[0].set_ylabel("probability")
fig.tight_layout()

## Summary

`multi_pauli_op` and `get_state` show that once a helper is written in terms of `from_operator_repr()` / `from_state_amplitudes()` and a size parameter `n_qubits`, it can build states and operators for any system size without any change to the code, and stays compatible with remote backends since nothing is tied to a particular register or device.